# 01 · Train the VAE  (Stage 1 — the latent space)

**Crowd-Driven Visual Generation.** Trains the `ConvVAE` on Colab's GPU over a **real art dataset (WikiArt)**, with optional **Weights & Biases** monitoring, and shows before/after reconstructions. The checkpoint gives the **latent space** the diffusion model (Stage 2) runs in.

> Runtime → Change runtime type → **GPU (T4 is plenty** — the models are small; save the A100).

## 1. Clone the repo & enter the project

In [ ]:
# Public repo over HTTPS. For a PRIVATE repo use a token:
#   !git clone --branch feature/crowd-driven-visual-generation https://<TOKEN>@github.com/nishant-kumar109/gen-ai-IISc.git
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation

In [ ]:
!pip install -q datasets wandb   # HF art dataset + W&B monitoring
import sys, torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Monitoring & credentials  *(fill these in before running)*

In [ ]:
# Prompts at runtime — keys are hidden and never saved in the notebook.
# Press Enter on either prompt to skip it.
import getpass

wandb_key = getpass.getpass("W&B API key (https://wandb.ai/authorize — Enter to skip): ").strip()
if wandb_key:
    import wandb; wandb.login(key=wandb_key); print("✓ W&B logged in — live loss curves on")
else:
    print("• W&B skipped — training still runs, no live curves")

hf_token = getpass.getpass("HF token (https://huggingface.co/settings/tokens — Enter to skip): ").strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token); print("✓ HuggingFace logged in — downloads un-throttled")
else:
    print("• HF token skipped — downloads may be slow/rate-limited")

## 3. (Optional) Persist checkpoints on Google Drive
Colab wipes local storage between sessions — mount Drive if you want the checkpoint to survive.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# OUT = '/content/drive/MyDrive/crowdgen/vae'
OUT = 'runs/vae'

## 4. Train on WikiArt
Streams the first `--limit` images (no full download). `--beta 0.25` lightens the KL pressure for **sharper reconstructions** (β=1.0 was over-smoothing). With `--wandb` you now get: per-step **loss/total, loss/recon, loss/kl**, **grad_norm**, **lr**, per-epoch averages, weight/gradient **histograms** (`wandb.watch`), and **live reconstruction images** every `--sample-every` epochs. GPU stats are under the run's **System** tab automatically.

Alternatives: a **folder** of your own images (`--dataset /content/art`), another HF art id, or `--dataset cifar10` for an instant smoke test.

Expected on **T4**: ~1–2 min streaming (with HF token) + ~3–5 min training.

In [ ]:
!python train_vae.py --dataset huggan/wikiart --limit 5000 --image-size 64 \
    --batch 128 --epochs 40 --lr 2e-4 --beta 0.25 --out {OUT} \
    --wandb --sample-every 5

## 5. Inspect reconstructions
Top row = real paintings, bottom row = VAE reconstructions (a bit soft — that's the VAE). If recognisable, the latent space is good for Stage 2. *(Also viewable live in your W&B run.)*

In [ ]:
from IPython.display import Image
Image(f'{OUT}/recon.png')

## Next
Checkpoint at `{OUT}/vae.pt`. **`02_train_diffusion.ipynb`** freezes this VAE, encodes images to latents, and trains the conditional DDPM (conditioned on each image's CLIP embedding). If you mounted Drive, note the path so Stage 2 can load it.